# 🔥 Never-AI — Super-Trainer GPU « Modèle 100 % Libre · Réseaux Sociaux (X / Web) » (v3.0)

Ce notebook pré-entraîne **`ChronoMorph-LM` (`Sprout-MoE` — ~72M paramètres, mémoire neuronale continue `Titans`, optimiseur `Muon`)** de façon **100 % libre et émergente** :

- **🚫 Zéro mode imposé, zéro prompt système, zéro consigne sur ce qu'il doit penser ou dire.**
- **𝕏 Immersion brute dans les réseaux sociaux** : vrais tweets français de **X / Twitter**, fils de réponses en direct (**X miroirs, microblogging FR temps réel, Reddit FR**), opinions spontanées du web francophone et culture générale.
- **🧠 Personnalité & humeur auto-construites** : ses réactions, son argot, son humour et ses variations d'état émergent **uniquement de ses propres poids synaptiques et de sa mémoire rapide `Titans`**, jamais d'une règle écrite à la main.

### Mode d'emploi :
1. Menu **Exécution** → **Modifier le type d'exécution** → **T4 GPU** → **Enregistrer**.
2. Cliquez sur **Exécution → Tout exécuter** (`Ctrl+F9`).
3. Importez **`never-ai-brain-sprout-moe.zip`** dans Never-AI Studio (**⚡ Super-Cortex → 📥 Importer `.zip` Colab**).

In [ ]:
# 1️⃣ VÉRIFICATION GPU & COFFRE LOCAL COLAB (ZÉRO BLOCAGE GOOGLE DRIVE !)
import os
import torch

TRY_GOOGLE_DRIVE = False
VAULT_DIR = '/content/NeverAI_Vault/checkpoints'
DRIVE_BACKUP_DIR = None

if TRY_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_BACKUP_DIR = '/content/drive/MyDrive/NeverAI_Vault/checkpoints'
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print('☁️ Google Drive connecté :', DRIVE_BACKUP_DIR)
    except Exception as err:
        print(f'ℹ️ Google Drive ignoré ({err}) -> SSD local Colab utilisé.')

os.makedirs(VAULT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_label = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (Activez T4 GPU dans Exécution -> Modifier le type d\'exécution !)'
print(f'✅ Accélérateur actif : {gpu_label} | PyTorch {torch.__version__}')

In [ ]:
# 2️⃣ CLONAGE NEVER-AI & INSTALLATION ULTRA-RAPIDE (~8 SECONDES)
import os, subprocess, sys

if not os.path.exists('/content/never-ai/.git'):
    subprocess.run([
        'git', 'clone', '-b', 'arena/403f5d7f-never-ai', '--depth', '1',
        'https://github.com/mouusieur/never-ai.git', '/content/never-ai'
    ], check=True)
else:
    subprocess.run(['git', '-C', '/content/never-ai', 'pull', '--ff-only'], check=False)

os.chdir('/content/never-ai')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers', 'safetensors', 'sympy', 'pyyaml', 'tqdm', 'datasets'
], check=True)

if '/content/never-ai/src' not in sys.path:
    sys.path.insert(0, '/content/never-ai/src')
print('✅ Never-AI v3.0 (Modèle Libre Sans Directive) prêt sur', os.getcwd())

In [ ]:
# 3️⃣ COLLECTE BRUTE SUR LES RÉSEAUX SOCIAUX (X / TWITTER, FILS EN DIRECT, REDDIT FR, WEB) — ZÉRO DIRECTIVE
import re
from datasets import load_dataset
from never_ai.core.special_tokens import BOS_TOKEN, EOS_TOKEN, NEVER_TOKEN, USER_TOKEN
from never_ai.data.social_stream import SocialStreamHarvester

raw_texts_for_tokenizer = []
formatted_docs = []

def clean_post(t: str) -> str:
    t = re.sub(r'https?://\S+', '', t or '').strip()
    t = re.sub(r'(?:@user\s*)+', '', t, flags=re.I).strip()
    return t

# A) VRAIS TWEETS X / TWITTER EN FRANÇAIS (texte brut uniquement — toutes les étiquettes sont ignorées !)
print('𝕏 Chargement de vrais posts X / Twitter francophones bruts...')
x_tweets = []
for ds_name, cfg_name in [
    ('cardiffnlp/tweet_sentiment_multilingual', 'french'),
    ('mteb/tweet_sentiment_multilingual', 'french'),
]:
    try:
        for split_name in ('train', 'validation', 'test'):
            ds_x = load_dataset(ds_name, cfg_name, split=split_name, streaming=True)
            for row in ds_x:
                tw = clean_post(row.get('text') or '')
                if len(tw) >= 12:
                    x_tweets.append(tw)
                    raw_texts_for_tokenizer.append(tw)
                    formatted_docs.append(f'{BOS_TOKEN}{tw}{EOS_TOKEN}')
            if len(x_tweets) >= 8000:
                break
        if len(x_tweets) >= 3000:
            break
    except Exception as e:
        print(f'   -> Info {ds_name}: {e}')
print(f'   -> {len(x_tweets):,} vrais tweets X francophones bruts chargés !')

# B) ASPIRATION EN DIRECT DES RÉSEAUX SOCIAUX (X miroirs + Microblogging FR temps réel + Fils Reddit FR)
print('📡 Aspiration en direct des posts et réponses sur les réseaux (X / Microblog FR / Reddit FR)...')
harvester = SocialStreamHarvester()
live_bundle = harvester.harvest_live_bundle(x_query='lang:fr', max_per_source=120)
formatted_docs.extend(live_bundle['sequences'])
raw_texts_for_tokenizer.extend(live_bundle['samples'])
print('   -> Flux live capturé :', live_bundle['counts'])

# C) AVIS, PUNCHLINES & RÉACTIONS SPONTANÉES D'INTERNAUTES FR (texte brut sans label)
print('🔥 Streaming de réactions spontanées d\'internautes francophones (Allociné / Web)...')
opinions_n = 0
try:
    ds_op = load_dataset('tblard/allocine', split='train', streaming=True)
    for row in ds_op:
        rev = clean_post(row.get('review') or '')
        if 25 <= len(rev) <= 900:
            raw_texts_for_tokenizer.append(rev)
            formatted_docs.append(f'{BOS_TOKEN}{rev}{EOS_TOKEN}')
            opinions_n += 1
            if opinions_n >= 5000:
                break
    print(f'   -> {opinions_n:,} prises de position et critiques spontanées chargées !')
except Exception as e:
    print(f'   -> Info Allociné ({e})')

# D) FILS DE DISCUSSIONS & ÉCHANGES HUMAINS BRUTS (OpenAssistant FR + Aya FR)
print('💬 Streaming de fils de discussion humains francophones...')
dialog_n = 0
try:
    ds_oasst = load_dataset('OpenAssistant/oasst2', split='train', streaming=True)
    by_id = {}
    for row in ds_oasst:
        if row.get('lang') != 'fr':
            continue
        mid, pid, role = row.get('message_id'), row.get('parent_id'), row.get('role')
        txt = (row.get('text') or '').strip()
        if not txt:
            continue
        by_id[mid] = txt
        # On enchaîne n'importe quel message parent -> réponse sans imposer de rôle !
        if pid in by_id:
            parent_txt = by_id[pid]
            raw_texts_for_tokenizer.append(parent_txt + '\n' + txt)
            formatted_docs.append(f'{BOS_TOKEN}{USER_TOKEN}{parent_txt}{NEVER_TOKEN}{txt}{EOS_TOKEN}')
            dialog_n += 1
            if dialog_n >= 5000:
                break
except Exception as e:
    print(f'   -> Info OASST2 ({e})')
print(f'   -> {dialog_n:,} échanges directs parent -> réponse chargés !')

# E) CONNAISSANCE DU MONDE (Wikipedia FR brut — sans consigne)
print('📚 Streaming de 4 000 articles Wikipedia FR pour la culture générale...')
ds_wiki = load_dataset('wikimedia/wikipedia', '20231101.fr', split='train', streaming=True)
wiki_n = 0
for row in ds_wiki:
    txt = (row.get('text') or '').strip()
    if len(txt) > 400:
        excerpt = txt[:2800]
        raw_texts_for_tokenizer.append(excerpt)
        formatted_docs.append(f'{BOS_TOKEN}{excerpt}{EOS_TOKEN}')
        wiki_n += 1
        if wiki_n >= 4000:
            break

total_chars = sum(len(t) for t in formatted_docs)
print(f'✅ Corpus 100 % brut prêt : {len(formatted_docs):,} séquences ({total_chars / 1e6:.1f} Mo — 0 directive, 0 mode imposé)')

In [ ]:
# 4️⃣ ENTRAÎNEMENT GPU LIBRE : CHRONOMORPH SPROUT-MOE (~72M PARAMS + TITANS + MUON)
import random, shutil, time
import torch
from never_ai.core.config import ChronoMorphConfig
from never_ai.core.model import ChronoMorphLM
from never_ai.core.tokenizer import MorphTokenizer
from never_ai.data.curriculum import ContinualDatasetAndReplay
from never_ai.optim.composite_muon import CompositeMuonOptimizer, WSMCheckpointMerger
from never_ai.vault.manager import VaultManager

# Montez STEPS à 3000 ou 6000 sur T4 GPU (~15 à 28 min) pour une construction encore plus riche !
STEPS = 2000
BATCH_SIZE = 12 if device == 'cuda' else 2
SEQ_LEN = 384 if device == 'cuda' else 128

print('🔤 Construction du vocabulaire neuronal BPE (vocab=8192) directement sur la langue des réseaux...')
tok = MorphTokenizer.train_from_iterator(raw_texts_for_tokenizer, vocab_size=8192)

cfg = ChronoMorphConfig.sprout_moe(vocab_size=tok.vocab_size)
model = ChronoMorphLM(cfg).to(device)
counts = model.count_parameters()
print(f"🧠 Modèle libre {cfg.stage_name} sur {device.upper()} : {counts['total_params']/1e6:.2f}M paramètres totaux ({counts['active_params']/1e6:.2f}M actifs/token)")

rng = random.Random()
rng.shuffle(formatted_docs)
split_idx = max(16, int(len(formatted_docs) * 0.97))
dataset = ContinualDatasetAndReplay(tok, seq_len=SEQ_LEN)
stats_ds = dataset.load_corpora(formatted_docs[:split_idx], formatted_docs[split_idx:])
print(f"📦 Séquences empaquetées : {stats_ds['train_chunks']:,} train | {stats_ds['val_chunks']:,} val")

opt = CompositeMuonOptimizer(model, muon_lr=3.0e-3, adamw_lr=8.0e-4, weight_decay=0.01, warmup_steps=80)
wsm = WSMCheckpointMerger(max_snapshots=6)
vault = VaultManager(vault_root=VAULT_DIR)
amp_dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32

t0 = time.time()
tokens_seen = 0
model.train()
for step in range(1, STEPS + 1):
    batch = dataset.sample_batch(batch_size=BATCH_SIZE, split='train', replay_ratio=0.20, device=device)
    opt.zero_grad()
    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
        out = model(batch.input_ids, targets=batch.targets, loss_mask=batch.loss_mask, update_titans=True)
        loss = out.loss
    loss.backward()
    opt.step(model)
    model.update_moe_biases()
    tokens_seen += int(batch.input_ids.numel())

    if step % 100 == 0 or step == 1:
        dt = max(1e-3, time.time() - t0)
        tok_s = tokens_seen / dt
        print(f'⚡ Step {step:4d}/{STEPS} | CE Loss = {out.ce_loss:.4f} | Vitesse = {tok_s:,.0f} tok/s | Tokens = {tokens_seen:,}')

    if step % 400 == 0 or step == STEPS:
        wsm.record_snapshot(model)
        consolidated = wsm.build_consolidated_state_dict(fallback_model=model)
        ckpt_path = vault.save_checkpoint(
            'latest',
            model,
            tok,
            metrics={
                'val_ce_loss': round(float(out.ce_loss), 4),
                'step': step,
                'total_tokens_trained': tokens_seen,
                'colab_gpu_trained': True,
            },
            consolidated_state_dict=consolidated,
        )
        if DRIVE_BACKUP_DIR:
            shutil.copytree(ckpt_path, os.path.join(DRIVE_BACKUP_DIR, 'latest'), dirs_exist_ok=True)
        print(f'💾 Checkpoint WSM sauvegardé -> {ckpt_path}')

In [ ]:
# 5️⃣ EXPRESSION LIBRE EN DIRECT (AVEC MÉMOIRE TITANS PERSISTANTE) + EXPORT .ZIP !
import os, zipfile
from never_ai.core.special_tokens import BOS_TOKEN, EOS_ID, NEVER_TOKEN, USER_ID, USER_TOKEN

model.eval()
stop_ids = {EOS_ID, USER_ID}
titans_state = None  # État neuronal interne persistant d'un tour à l'autre (zéro prompt système !)

print('🗣️ Discussion libre sans aucune consigne (état interne Titans continu) :\n' + '=' * 70)
for msg in [
    'ça dit quoi ? tu penses à quoi là ?',
    'les gens sur X ils abusent parfois non ?',
]:
    p_ids = tok.encode(f'{BOS_TOKEN}{USER_TOKEN}{msg}{NEVER_TOKEN}')
    gen_ids, titans_state = model.generate(
        p_ids,
        max_new_tokens=90,
        temperature=0.75,
        top_p=0.92,
        stop_token_ids=stop_ids,
        titans_states=titans_state,
    )
    reply = tok.decode(gen_ids[len(p_ids):], skip_special_tokens=True).strip()
    print(f'👤 Vous     : {msg}')
    print(f'🤖 Never-AI : {reply}\n')

zip_out = '/content/never-ai-brain-sprout-moe.zip'
latest_dir = os.path.join(VAULT_DIR, 'latest')
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(latest_dir)):
        fpath = os.path.join(latest_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)

size_mb = os.path.getsize(zip_out) / (1024 * 1024)
print(f'📦 Archive prête : {zip_out} ({size_mb:.1f} Mo)')
print('👉 Dans Never-AI Studio : ⚡ Super-Cortex -> 📥 Importer .zip Colab !')

try:
    from google.colab import files
    files.download(zip_out)
except Exception as e:
    print(f'ℹ️ Téléchargement disponible dans le panneau Fichiers (icône 📁 à gauche) : {zip_out} ({e})')